# Train a ConvVAE on CelebA (Colab + KaggleHub)

This notebook uses the connected Kaggle dataset in Colab through KaggleHub, prepares an ImageFolder dataset, and trains a ConvVAE.

Google Drive is optional and is only used if you want checkpoints and extracted images to persist across sessions.


In [ ]:
# Colab dependency setup
!pip -q install torch torchvision tqdm matplotlib kagglehub

In [ ]:
# Optional: persist data/checkpoints on Google Drive
from pathlib import Path
from google.colab import drive

USE_GDRIVE = True

if USE_GDRIVE:
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/celeba_vae_project')
else:
    BASE_DIR = Path('/content')

BASE_DIR.mkdir(parents=True, exist_ok=True)
print('BASE_DIR =', BASE_DIR)

In [ ]:
# Resolve CelebA source from the connected Kaggle dataset and prepare ImageFolder structure
from pathlib import Path
import os
import shutil
import zipfile

import kagglehub

base_dir = globals().get('BASE_DIR', Path('/content'))
dataset_root = base_dir / 'local_datasets' / 'celeba'
image_dir = dataset_root / 'img_align_celeba'
dataset_root.mkdir(parents=True, exist_ok=True)

def _find_image_dir(source_root: Path):
    """Search for the folder that directly contains *.jpg images."""
    candidates = [
        source_root / 'img_align_celeba' / 'img_align_celeba',  # Kaggle nested structure
        source_root / 'img_align_celeba',
        source_root / 'celeba-dataset' / 'img_align_celeba' / 'img_align_celeba',
        source_root / 'celeba-dataset' / 'img_align_celeba',
    ]
    for candidate in candidates:
        if candidate.exists() and any(candidate.glob('*.jpg')):
            print('Found images at:', candidate)
            return candidate

    # Debug: print what's actually there
    print('Contents of source root:')
    for p in sorted(source_root.rglob('*'))[:30]:
        print(' ', p)

    raise RuntimeError(
        f'Could not locate jpg images under {source_root}. See contents above.'
    )

if not image_dir.exists() or not any(image_dir.glob('*.jpg')):
    source_root = Path(kagglehub.dataset_download('jessicali9530/celeba-dataset'))
    print('KaggleHub source:', source_root)

    src_images = _find_image_dir(source_root)

    # Symlink instead of copying to avoid moving 200k files
    if image_dir.is_symlink() or image_dir.exists():
        if image_dir.is_symlink():
            image_dir.unlink()
        else:
            shutil.rmtree(str(image_dir))
    os.symlink(src_images, image_dir)
    print('Symlinked:', image_dir, '->', src_images)

print('Image directory:', image_dir)
print('JPG files:', len(list(image_dir.glob('*.jpg'))))

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Training config
IMAGE_SIZE = 64
BATCH_SIZE = 128
LATENT_DIM = 128
EPOCHS = 20
LR = 1e-3
BETA = 1e-4
NUM_WORKERS = 2

base_dir = globals().get('BASE_DIR', Path('/content'))

# Colab paths
DATA_ROOT = base_dir / 'local_datasets' / 'celeba'
IMAGE_DIR = DATA_ROOT / 'img_align_celeba'
CKPT_DIR = base_dir / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f'Image directory not found: {IMAGE_DIR}. Run Kaggle setup cells first.')

print('DATA_ROOT =', DATA_ROOT)
print('CKPT_DIR =', CKPT_DIR)

In [ ]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_ds = datasets.ImageFolder(
    root=str(DATA_ROOT),
    transform=transform,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

x_img, x_class = train_ds[0]
print('data_root =', DATA_ROOT)
print('number of images:', len(train_ds))
print('classes:', train_ds.classes)
print('class_to_idx:', train_ds.class_to_idx)
print('x_img shape:', tuple(x_img.shape))
print('x_class:', x_class)
print('Batches per epoch:', len(train_loader))

In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, in_channels=3, image_size=64, latent_dim=128):
        super().__init__()
        if image_size % 16 != 0:
            raise ValueError('image_size must be divisible by 16')

        self.in_channels = in_channels
        self.image_size = image_size
        self.latent_dim = latent_dim

        self.enc_spatial = image_size // 16
        self.enc_feat_dim = 256 * self.enc_spatial * self.enc_spatial

        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )

        self.fc_mu = nn.Linear(self.enc_feat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.enc_feat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.enc_feat_dim)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, in_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def encode(self, x):
        h = self.encoder(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z).view(z.size(0), 256, self.enc_spatial, self.enc_spatial)
        return self.decoder(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


def vae_loss(x, x_hat, mu, logvar, beta=1e-4):
    recon = F.mse_loss(x_hat, x, reduction='mean')
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon + beta * kld
    return total, recon.detach(), kld.detach()


model = ConvVAE(in_channels=3, image_size=IMAGE_SIZE, latent_dim=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
def denorm(x):
    return x.clamp(-1, 1).add(1).div(2)


def show_batch_recons(model, loader, device, n=8):
    model.eval()
    with torch.no_grad():
        x, _ = next(iter(loader))
        x = x[:n].to(device)
        x_hat, _, _ = model(x)

    x_vis = denorm(x.cpu())
    xh_vis = denorm(x_hat.cpu())

    grid = utils.make_grid(torch.cat([x_vis, xh_vis], dim=0), nrow=n)
    plt.figure(figsize=(2.0 * n, 4))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis('off')
    plt.title('Top: originals | Bottom: reconstructions')
    plt.show()


def train_vae(model, loader, optimizer, device, epochs=20, beta=1e-4):
    history = []
    model.train()

    for epoch in range(1, epochs + 1):
        total_loss, total_recon, total_kld = 0.0, 0.0, 0.0
        pbar = tqdm(loader, desc=f'Epoch {epoch}/{epochs}')
        t0 = time.time()

        for x, _ in pbar:
            x = x.to(device, non_blocking=True)
            x_hat, mu, logvar = model(x)
            loss, recon, kld = vae_loss(x, x_hat, mu, logvar, beta=beta)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += float(loss.item())
            total_recon += float(recon.item())
            total_kld += float(kld.item())

            pbar.set_postfix(loss=f'{loss.item():.4f}', recon=f'{recon.item():.4f}', kld=f'{kld.item():.4f}')

        n = max(1, len(loader))
        stats = {
            'epoch': epoch,
            'loss': total_loss / n,
            'recon': total_recon / n,
            'kld': total_kld / n,
            'sec': time.time() - t0,
        }
        history.append(stats)

        ckpt_path = CKPT_DIR / f'vae_celeba_epoch_{epoch:03d}.pt'
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'stats': stats,
            'config': {
                'image_size': IMAGE_SIZE,
                'latent_dim': LATENT_DIM,
                'beta': BETA,
                'batch_size': BATCH_SIZE,
                'lr': LR,
            },
        }, ckpt_path)

        print(
            f"Epoch {epoch:02d} | loss={stats['loss']:.5f} recon={stats['recon']:.5f} "
            f"kld={stats['kld']:.5f} | {stats['sec']:.1f}s | saved {ckpt_path.name}"
        )

    return history

In [ ]:
history = train_vae(
    model=model,
    loader=train_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS,
    beta=BETA,
)

show_batch_recons(model, train_loader, DEVICE, n=8)

In [ ]:
losses = [h['loss'] for h in history]
recons = [h['recon'] for h in history]
klds = [h['kld'] for h in history]

plt.figure(figsize=(10, 4))
plt.plot(losses, label='loss')
plt.plot(recons, label='recon')
plt.plot(klds, label='kld')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('Training curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('Latest checkpoint directory:', CKPT_DIR)
print('Example checkpoint:', CKPT_DIR / f'vae_celeba_epoch_{EPOCHS:03d}.pt')